# Разведочный анализ датасета Дрёмова для категоризатора

Наш текущий категоризатор обучен на синтетических данных и показывает почти
идеальные метрики, но это переобучение на собственный словарь и на реальных
чеках он так работать не будет. Здесь мы изучаем реальный датасет (Дрёмов, ~14k
размеченных чековых позиций с категориями), чтобы переобучить категоризатор на
настоящих данных и получить честные метрики.

Главный вопрос этого EDA это категории. Наши шесть категорий заданы проектом:
Продукты, Кафе и рестораны, Транспорт, Аптека, Развлечения, Прочее. Категории в
датасете Дрёмова почти наверняка другие, и прежде чем обучать, нужно понять их
схему и построить маппинг в наши шесть. Без этого обучение невозможно.

Также смотрим: формат текста позиции, баланс классов, длины, качество данных.

## Что в папке

Сначала смотрим, какие файлы скачались и в каком формате.

In [24]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

def find_root(start):
    p = Path(start).resolve()
    for c in [p, *p.parents]:
        if (c / "src").exists() or (c / "data").exists():
            return c
    return p

root = find_root(Path.cwd())
folder = root / "data" / "raw" / "dremov"

print("Файлы в папке:")
for f in folder.iterdir():
    print("  ", f.name, f"({f.stat().st_size // 1024} KB)")

Файлы в папке:
   train.csv (1148 KB)
   train_checks.csv (106 KB)


## Загрузка и первый взгляд

В папке два файла: train.csv (крупный, вероятно позиции с категориями) и
train_checks.csv (мелкий, возможно метаданные чеков). Смотрим структуру обоих,
чтобы понять, что где лежит и какой файл нам нужен для обучения категоризатора.

In [25]:
train = pd.read_csv(folder / "train.csv")
checks = pd.read_csv(folder / "train_checks.csv")

print("train.csv ")
print("Размер:", train.shape)
print("Колонки:", train.columns.tolist())
print("Пропуски:\n", train.isna().sum())
print("\nПервые 10 строк:")
print(train.head(10).to_string())

print("\n\n train_checks.csv")
print("Размер:", checks.shape)
print("Колонки:", checks.columns.tolist())
print("\nПервые 5 строк:")
print(checks.head(5).to_string())

train.csv 
Размер: (13682, 5)
Колонки: ['check_id', 'name', 'category', 'price', 'count']
Пропуски:
 check_id    0
name        1
category    0
price       0
count       0
dtype: int64

Первые 10 строк:
   check_id                                      name       category  price  count
0         0  *3479755 ТRUF.Конф.кр.корп.гл.вк.шок180г  Чай и сладкое   49.0    2.0
1         0   3408392 ECONTA Мешки д/мусора 30л  30шт       Для дома   21.0    1.0
2         0     3260497 ЯШКИНО Рулет С ВАР.СГУЩ. 200г  Чай и сладкое   39.0    1.0
3         0           3300573 Пакет ПЯТЕРОЧКА 65х40см       Упаковка    4.0    1.0
4         0      3413607 ЗЕР/СЕЛ.Сухари с изюмом 250г  Чай и сладкое   35.0    1.0
5         0   3221388 ШАРЛ.Печенье вафел.рассыпч.225г  Чай и сладкое   38.0    1.0
6         0              *97452 ПРОСТ.Кефир 3,2% 930г        Молочка   55.0    1.0
7         0     57575 MILFORD Зам.сахара доз.  650таб        Бакалея  119.0    1.0
8         0      29880 ПИСК.Ацидоб.2.2%сл.пюр-пак0.

### Что в данных

Два связанных файла. `train.csv`: 13682 позиции с категориями: текст позиции
(name), категория, цена, количество, и check_id - ссылка на чек. `train_checks.csv`
- 2042 чека с магазином, датой и суммой. Связаны по check_id: позиции группируются
в чеки.

Для категоризатора нам нужен в первую очередь train.csv. Это ровно наш вход. Магазин из train_checks можно подмешать как
дополнительный сигнал, но базово хватит и текста позиции.

Текст позиции сырой, как в чеке: с артикулами, сокращениями, в разном регистре
(«ПРОСТ.Кефир 3,2%», «ЯШКИНО Рулет»). Это хорошо т.к. именно такой текст придёт от OCR
в реальном пайплайне, модель учится на реалистичных данных, а не на чистых.

Пропусков почти нет (одно пустое name). Категория есть у всех.

## Категории: какие есть и как мапить в наши шесть

Категории Дрёмова это своя схема, и их больше наших шести. Чтобы обучать наш
категоризатор, нужно свести их к нашим: Продукты, Кафе и рестораны, Транспорт,
Аптека, Развлечения, Прочее. Сначала смотрим полный список категорий датасета и
их распределение и по нему построим маппинг.

In [26]:
print("Всего уникальных категорий:", train["category"].nunique())
print("\nРаспределение категорий:")
print(train["category"].value_counts())

Всего уникальных категорий: 25

Распределение категорий:
category
Овощи и фрукты    1761
Чай и сладкое     1485
Молочка           1415
Для дома          1271
Бакалея           1051
Гастрономия        878
Хлеб               652
Упаковка           628
Мясо и птица       600
Напитки            567
Дети               503
Кафе               422
Здоровье           375
Алкоголь           343
Снеки              332
Кулинария          235
Косметика          230
Животные           191
Одежда и обувь     190
Услуги             140
Машина             118
Табак               99
Рыба                75
Не определена       70
Компьютер           51
Name: count, dtype: int64


Дрёмов хорошо покрывает Продукты, но почти не покрывает Развлечения, Транспорт,
Кафе, Аптеку. Эти категории добираем синтетикой (генератор из ноутбука 04).
Прикинем целевой баланс: сколько реальных примеров берём из Дрёмова и сколько
синтетических нужно догенерировать по каждой категории, чтобы классы были
сопоставимы и ни один не был вырожденным.

In [28]:
from pathlib import Path
import pandas as pd

def find_root(start):
    p = Path(start).resolve()
    for c in [p, *p.parents]:
        if (c / "src").exists() or (c / "data").exists():
            return c
    return p

root = find_root(Path.cwd())
train = pd.read_csv(root / "data" / "raw" / "dremov" / "train.csv")

mapping = {
    "Овощи и фрукты": "Продукты", "Чай и сладкое": "Продукты", "Молочка": "Продукты",
    "Бакалея": "Продукты", "Гастрономия": "Продукты", "Хлеб": "Продукты",
    "Мясо и птица": "Продукты", "Напитки": "Продукты", "Снеки": "Продукты",
    "Кулинария": "Продукты", "Рыба": "Продукты", "Алкоголь": "Продукты", "Табак": "Продукты",
    "Кафе": "Кафе и рестораны", "Здоровье": "Аптека", "Машина": "Транспорт",
    "Для дома": "Прочее", "Упаковка": "Прочее", "Дети": "Прочее", "Косметика": "Прочее",
    "Животные": "Прочее", "Одежда и обувь": "Прочее", "Услуги": "Прочее",
    "Компьютер": "Прочее", "Не определена": "Прочее",
}
train["our_cat"] = train["category"].map(mapping)

per_check = train.groupby("check_id")["our_cat"].agg(["nunique", lambda x: x.value_counts().index[0], "count"])
per_check.columns = ["n_categories", "dominant", "n_items"]

print("Категорий на чек (распределение):")
print(per_check["n_categories"].value_counts().sort_index())
print("\nДоминирующая категория чека:")
print(per_check["dominant"].value_counts())
print("\nСредне позиций в чеке:", round(per_check["n_items"].mean(), 1))

Категорий на чек (распределение):
n_categories
1    1213
2     716
3     106
4       6
Name: count, dtype: int64

Доминирующая категория чека:
dominant
Продукты            1285
Прочее               472
Кафе и рестораны     106
Транспорт             90
Аптека                88
Name: count, dtype: int64

Средне позиций в чеке: 6.7


## Очистка дрёмовских данных через NER

Ключевой момент: в реальном пайплайне категоризатор получает товары не сырыми из
чека, а после NER уже извлечённые чистые товары и бренды. Поэтому обучать его на
сырых дрёмовских названиях с артикулами неправильно: формат обучения разойдётся с
продакшеном, и модель будет цепляться за формат, а не за смысл.

Прогоняем дрёмовские позиции через наш NER и собираем сигнатуры из извлечённых
товаров и брендов. Так обучающие данные
категоризатора совпадут с тем, что он реально получит на инференсе.

In [30]:
from tqdm import tqdm
from src.models.ner_extractor import NERExtractor
ner = NERExtractor()

def ner_signature(shop, names):
    parts = []
    for name in names:
        res = ner.extract(str(name))
        chunk = " ".join(res["goods"] + res["brands"]).strip().lower()
        if chunk:
            parts.append(chunk)
    head = shop if shop and str(shop).lower() not in ("не известно", "nan", "") else ""
    if not parts:
        return None
    return (f"{head}. " + ", ".join(parts)) if head else ", ".join(parts)

groups = list(train.groupby("check_id"))
dremov_rows = []
for check_id, grp in tqdm(groups):
    info = per_check_cat.loc[check_id]
    cat = info["dominant"]
    if cat not in ("Продукты", "Прочее") or info["n_cat"] > 2:
        continue
    shop = shop_by_check.get(check_id, "")
    names = [n for n in grp["name"].head(6) if pd.notna(n)]
    sig = ner_signature(shop, names)
    if sig:
        dremov_rows.append({"text": sig, "label": cat})

print("Готово:", len(dremov_rows))

  0%|          | 0/2041 [00:00<?, ?it/s]

2026-06-11 12:12:27 | INFO    | src.models.ner_extractor | NER загружен на cpu


100%|██████████| 2041/2041 [05:10<00:00,  6.56it/s]

Готово: 1587


In [31]:
import pandas as pd
dremov_df = pd.DataFrame(dremov_rows)
print(dremov_df["label"].value_counts())
print()
for r in dremov_rows[:6]:
    print(f"[{r['label']}] {r['text']}")

label
Продукты    1178
Прочее       409
Name: count, dtype: int64

[Продукты] тruf, мешки econta, рулет яшкино, пакет, сухари, печенье
[Прочее] ЕВРОПА. яйцо, сок фрутоняня, помело, нектар сочная долина
[Продукты] ИП Роздухов М. Е.. молоко, mоkka
[Продукты] Агроторг. мандарины, яйца
[Продукты] Агроторг. апельсины выгодно, йогурт, сухари, шокол бабаевский, шокол бабаевский, eco - bot
[Продукты] Тандер. масло вкуснотеево, батон тамбовский, конфеты, праздник, молоко сметанин, жаклин


По итогам анализа данные распределяются так: Дрёмов хорошо покрывает Продукты и
Прочее, но почти не покрывает Кафе, Транспорт, Аптеку и
вообще не содержит Развлечений. Поэтому делаем гибрид:

- Продукты и Прочее берём из Дрёмова;
- Кафе, Транспорт, Аптека, Развлечения  из синтетики (там категории чёткие, а
  тут «Здоровье/Кафе» смыслово расходятся с нашими и малочисленны);
- Прочее добиваем синтетикой до баланса.

Оба источника приведены к единому формату «магазин. товары» т.е. ровно к тому, что
соберёт пайплайн на инференсе.

Сначала генерируем синтетику тем же генератором, что использовался в ноутбуке 04.

In [32]:
import sys
sys.path.insert(0, str(root))
from src.data.datasets import generate_categorizer_dataset

synth = generate_categorizer_dataset(per_class=1200, seed=42)
synth_df = pd.DataFrame(synth)
print("Синтетики сгенерировано:", len(synth_df))
print(synth_df["label"].value_counts())
print("\nПримеры:")
for r in synth[:4]:
    print(f"  [{r['label']}] {r['text']}")

Синтетики сгенерировано: 7200
label
Прочее              1200
Кафе и рестораны    1200
Транспорт           1200
Продукты            1200
Аптека              1200
Развлечения         1200
Name: count, dtype: int64

Примеры:
  [Прочее] Икеа. мяч, книга, блокнот, лопата, ручка шариковая, скотч
  [Кафе и рестораны] Сбарро. картофель фри, блинчики, сэндвич, сет суши, салат цезарь
  [Транспорт] Яндекс Такси. бензин АИ-98, поездка такси
  [Продукты] Перекрёсток. печенье, сахар, колбаса


Собираем обучающий набор по плану: Продукты из Дрёмова целиком, Прочее из Дрёмова
плюс синтетика до баланса, остальные четыре категории из синтетики.

In [33]:
import sys
sys.path.insert(0, str(root))
from src.data.datasets import generate_categorizer_dataset

synth = generate_categorizer_dataset(per_class=1200, seed=42)
synth_df = pd.DataFrame(synth)

target = {
    "Продукты": {"dremov": 1178, "synth": 0},
    "Прочее": {"dremov": 409, "synth": 800},
    "Кафе и рестораны": {"dremov": 0, "synth": 1200},
    "Транспорт": {"dremov": 0, "synth": 1200},
    "Аптека": {"dremov": 0, "synth": 1200},
    "Развлечения": {"dremov": 0, "synth": 1200},
}

mixed = []
for cat_name, plan in target.items():
    if plan["dremov"] > 0:
        mixed.extend(dremov_df[dremov_df["label"] == cat_name].head(plan["dremov"]).to_dict("records"))
    if plan["synth"] > 0:
        mixed.extend(synth_df[synth_df["label"] == cat_name].head(plan["synth"]).to_dict("records"))

mixed_df = pd.DataFrame(mixed).sample(frac=1, random_state=42).reset_index(drop=True)
print("Смесь:", len(mixed_df))
print(mixed_df["label"].value_counts())

Смесь: 7187
label
Прочее              1209
Кафе и рестораны    1200
Аптека              1200
Транспорт           1200
Развлечения         1200
Продукты            1178
Name: count, dtype: int64


## Сохранение датасета

Делим смесь на train/val/test (80/10/10) со стратификацией по категории, чтобы во
всех частях баланс классов сохранился. Сохраняем в JSONL тот же формат, что
читает обучающий код категоризатора. Этот датасет заменит синтетику из ноутбука 04
при переобучении.

In [34]:
from sklearn.model_selection import train_test_split
import json

train_df, temp_df = train_test_split(mixed_df, test_size=0.2, random_state=42, stratify=mixed_df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

out_dir = root / "data" / "processed" / "categorizer_real"
out_dir.mkdir(parents=True, exist_ok=True)
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    with (out_dir / f"{name}.jsonl").open("w", encoding="utf-8") as f:
        for _, row in part.iterrows():
            f.write(json.dumps({"text": row["text"], "label": row["label"]}, ensure_ascii=False) + "\n")
    print(f"{name}: {len(part)}")

train: 5749
validation: 719
test: 719


In [36]:
import cv2
from pathlib import Path
from src.models.pipeline import ReceiptPipeline

# найдём какое-нибудь тестовое фото чека
candidates = list((root / "data").rglob("*.jpg")) + list((root / "data").rglob("*.png"))
print("Найдено изображений:", len(candidates))
for c in candidates[:10]:
    print("  ", c.relative_to(root))

Найдено изображений: 1574
   data\processed\detector\test\images\1002-receipt_jpg.rf.2511d340757d6c5aff57375fed6f652c.jpg
   data\processed\detector\test\images\1006-receipt_jpg.rf.45b45df0511f77d9654fc6b017b3f1e2.jpg
   data\processed\detector\test\images\1008-receipt_jpg.rf.ef97d80e17c42a4818729193b2fcedb9.jpg
   data\processed\detector\test\images\1014-receipt_jpg.rf.63599b272ee848286492d145ec719ed2.jpg
   data\processed\detector\test\images\1020-receipt_jpg.rf.a12022059a3021662c1bd3bf74aaff96.jpg
   data\processed\detector\test\images\1024-receipt_jpg.rf.1501726427ab8f8c60ae63f9dd0eb2ed.jpg
   data\processed\detector\test\images\1067-receipt_jpg.rf.82968ad2b86e73d1a110627679795a08.jpg
   data\processed\detector\test\images\1073-receipt_jpg.rf.db8befe5c6804c72dc2d4a78cfabb73a.jpg
   data\processed\detector\test\images\1074-receipt_jpg.rf.969ca79e357dfa3650bf7efd9542c16d.jpg
   data\processed\detector\test\images\1075-receipt_jpg.rf.39032df8e863ae0d0b580d7914620f06.jpg


In [35]:
import cv2
from src.models.pipeline import ReceiptPipeline

img_path = candidates[0]
image = cv2.imread(str(img_path))
print("Фото:", img_path.name, "размер:", image.shape)

pipe = ReceiptPipeline()
result = pipe.process(image)

print("\nМагазин:   ", result["merchant"])
print("Дата:      ", result["date"])
print("Сумма:     ", result["total"])
print("Язык:      ", result["language"])
print("Категория: ", result["receipt_type"])
print("Статус:    ", result["status"])
print("Позиции:")
for it in result["items"]:
    print(f"   {it['good']} | бренд: {it['brand']}")

========== OCR ==========
"""Обёртка над OCR-движком (PaddleOCR).

Распознаёт текст и его координаты на изображении чека. Используется как
вспомогательный сигнал и как резервный путь извлечения полей по правилам,
если основная модель извлечения выдаёт невалидный результат.
Движок грузится один раз при первом вызове.
"""
from __future__ import annotations

from src.utils.logging import get_logger

logger = get_logger(__name__)


class OCREngine:
    def __init__(self, lang="en"):
        self.lang = lang
        self._engine = None

    def _load(self):
        if self._engine is None:
            from paddleocr import PaddleOCR

            logger.info("Загрузка PaddleOCR (lang=%s)", self.lang)
            self._engine = PaddleOCR(use_angle_cls=True, lang=self.lang, show_log=False)
        return self._engine

    def recognize(self, image):
        """Распознаёт текст на изображении (numpy BGR или путь).

        Возвращает список словарей: text, confidence, box (4 точки).
        """